# Bulk RNA-seq time course

Figures 1B, 1D, 3A, 4A, S11; DESeq2 results used for Tables S1, S2, S4 and Figures S3, S4

Environment: `bulk_R_env` (`envs/bulk_R_env.yml`)

Inputs:
- `bulk_RNAseq_htseq_raw_counts.csv` (GSE348942), in `data_dir/GSE348942/`
- `data/bulk_RNAseq_sample_metadata.csv`

In [ ]:
suppressPackageStartupMessages({
  library(yaml)
  library(DESeq2)
  library(ggplot2)
  library(patchwork)
  library(matrixStats)
  library(scales)
})

ROOT <- ".."
config <- read_yaml(file.path(ROOT, "config.yaml"))
DATA_DIR <- file.path(config$data_dir, "GSE348942")
OUT_DIR <- config$out_dir
FIG_DIR <- file.path(OUT_DIR, "figures")
TAB_DIR <- file.path(OUT_DIR, "tables")
SRC_DIR <- file.path(OUT_DIR, "source_data")
DE_DIR  <- file.path(OUT_DIR, "bulk_deseq2")
for (d in c(FIG_DIR, TAB_DIR, SRC_DIR, DE_DIR)) dir.create(d, recursive = TRUE, showWarnings = FALSE)

palette <- c("#1F77B4", "#FF7F0E")
times <- c(0, 5, 10, 15, 45, 180)

In [ ]:
counts <- read.csv(file.path(DATA_DIR, "bulk_RNAseq_htseq_raw_counts.csv"),
                   row.names = 1, check.names = FALSE)
counts <- counts[!startsWith(rownames(counts), "__"), ]

meta <- read.csv(file.path(ROOT, "data", "bulk_RNAseq_sample_metadata.csv"))
rownames(meta) <- meta$sample
meta$line <- factor(meta$line)
meta$infected <- factor(meta$infected, levels = c("no", "yes"))
meta <- meta[colnames(counts), ]
stopifnot(identical(rownames(meta), colnames(counts)))

In [ ]:
meta$time_scaled <- as.numeric(scale(meta$time))

dds <- DESeqDataSetFromMatrix(countData = counts, colData = meta,
                              design = ~ line + time_scaled + infected)
dds <- DESeq(dds)
rld <- rlog(dds)
norm_counts <- counts(dds, normalized = TRUE)

write.csv(norm_counts, file.path(OUT_DIR, "bulk_RNAseq_normalized_counts.csv"))

In [ ]:
gene_counts <- function(genes) {
  out <- lapply(genes, function(g) {
    d <- plotCounts(dds, gene = g, intgroup = c("infected", "time"),
                    returnData = TRUE, normalized = TRUE)
    d$time <- as.numeric(as.character(d$time))
    d$gene <- g
    d$sample <- rownames(d)
    d
  })
  do.call(rbind, out)
}

axis_time <- function(x) {
  ifelse(x <= 15, x,
         ifelse(x <= 45, 15 + (x - 15) * 10 / 30,
                25 + (x - 45) * 15 / 135))
}

frame_layer <- function(levels_vec, facet_var, x_limits, lw) {
  fd <- data.frame(level = factor(levels_vec, levels = levels_vec),
                   x_min = x_limits[1], x_max = x_limits[2])
  names(fd)[1] <- facet_var
  list(
    geom_segment(data = fd, aes(x = x_min, xend = x_max, y = Inf, yend = Inf),
                 inherit.aes = FALSE, color = "black", linewidth = lw),
    geom_segment(data = fd, aes(x = x_min, xend = x_max, y = -Inf, yend = -Inf),
                 inherit.aes = FALSE, color = "black", linewidth = lw),
    geom_segment(data = fd, aes(x = x_min, xend = x_min, y = -Inf, yend = Inf),
                 inherit.aes = FALSE, color = "black", linewidth = lw),
    geom_segment(data = fd, aes(x = x_max, xend = x_max, y = -Inf, yend = Inf),
                 inherit.aes = FALSE, color = "black", linewidth = lw)
  )
}

save_fig <- function(p, name, w, h, dpi) {
  ggsave(file.path(FIG_DIR, paste0(name, ".png")), p, width = w, height = h,
         units = "in", dpi = dpi, limitsize = FALSE, bg = "white")
  ggsave(file.path(FIG_DIR, paste0(name, ".pdf")), p, width = w, height = h,
         units = "in", limitsize = FALSE, bg = "white", device = cairo_pdf)
}

x_breaks <- axis_time(times)
x_pad <- diff(range(x_breaks)) * 0.035
x_limits <- c(min(x_breaks) - x_pad, max(x_breaks) + x_pad)

## Figure 1B

In [ ]:
d <- gene_counts(c("MS2", "MS2_negative"))
d <- d[d$infected == "yes", ]

axis_time_1b <- function(x) {
  ifelse(x <= 15, x * 8 / 5,
         ifelse(x <= 45, 24 + (x - 15) * 14 / 30,
                38 + (x - 45) * 18 / 135))
}
d$x_time <- axis_time_1b(d$time)
xb <- axis_time_1b(times)
xp <- diff(range(xb)) * 0.035

decade_breaks <- function(lims) 10^seq(ceiling(log10(lims[1])), floor(log10(lims[2])), by = 1)

p <- ggplot(d, aes(x = x_time, y = count, linetype = gene, group = gene)) +
  geom_point(size = 4, color = palette[2]) +
  stat_summary(fun = "mean", geom = "line", linewidth = 2.5, color = palette[2]) +
  stat_summary(fun.data = mean_se, geom = "errorbar", linetype = "solid",
               width = 0.9, linewidth = 1.3, color = palette[2]) +
  scale_linetype_manual(values = c(MS2 = "solid", MS2_negative = "33"),
                        labels = c(MS2 = "MS2 (+)", MS2_negative = "MS2 (\u2212)")) +
  scale_x_continuous(limits = c(min(xb) - xp, max(xb) + xp), breaks = xb,
                     labels = times, expand = expansion(0, 0)) +
  scale_y_log10(breaks = decade_breaks,
                labels = trans_format("log10", math_format(10^.x))) +
  labs(x = "Time (minutes post infection)", y = "Normalized count (log scale)", linetype = NULL) +
  theme_minimal() +
  theme(
    panel.border = element_rect(fill = NA, color = "black", linewidth = 1),
    panel.grid.minor = element_blank(),
    axis.title.x = element_text(size = 30, margin = margin(t = 12)),
    axis.title.y = element_text(size = 30, margin = margin(r = 12)),
    axis.text = element_text(size = 28, color = "black"),
    axis.ticks = element_line(color = "black", linewidth = 1.2),
    axis.ticks.length = unit(0.3, "cm"),
    legend.text = element_text(size = 30, color = "black"),
    legend.key.size = unit(1.1, "cm"),
    legend.key.width = unit(1.8, "cm")
  ) +
  guides(linetype = guide_legend(override.aes = list(color = palette[2]))) +
  coord_cartesian(clip = "off")

options(repr.plot.width = 8.5, repr.plot.height = 7)
print(p)
save_fig(p, "Fig1B", 10, 8.5, 800)
write.csv(d[, c("sample", "gene", "time", "infected", "count")], file.path(SRC_DIR, "Fig1B_data.csv"), row.names = FALSE)

## Figure 1D

In [ ]:
rv <- rowVars(assay(rld))
top <- order(rv, decreasing = TRUE)[seq_len(min(500, length(rv)))]
pca <- prcomp(t(assay(rld)[top, ]))
pct <- round(100 * pca$sdev^2 / sum(pca$sdev^2))

pc <- data.frame(sample = colnames(rld), pca$x[, 1:4],
                 time = factor(rld$time, levels = times),
                 infected = factor(rld$infected, levels = c("no", "yes"),
                                   labels = c("Control (LB)", "MS2 (MOI=10)")))

shapes <- c("0" = 8, "5" = 4, "10" = 15, "15" = 18, "45" = 16, "180" = 17)
cols <- setNames(palette, levels(pc$infected))

pca_plot <- function(xpc, ypc) {
  ggplot(pc, aes(x = .data[[xpc]], y = .data[[ypc]], color = infected, shape = time)) +
    geom_point(size = 7, alpha = 0.85, stroke = 1.8) +
    labs(x = paste0(xpc, ": ", pct[as.integer(sub("PC", "", xpc))], "% variance"),
         y = paste0(ypc, ": ", pct[as.integer(sub("PC", "", ypc))], "% variance")) +
    scale_color_manual(values = cols, name = "Infection") +
    scale_shape_manual(values = shapes, name = "Time") +
    scale_x_continuous(expand = expansion(mult = 0.15)) +
    scale_y_continuous(expand = expansion(mult = 0.15)) +
    guides(color = guide_legend(byrow = TRUE, order = 1, override.aes = list(size = 9, shape = 16)),
           shape = guide_legend(byrow = TRUE, order = 2, override.aes = list(size = 9, color = "black"))) +
    theme_classic(base_size = 18) +
    theme(
      axis.line = element_blank(),
      panel.border = element_rect(color = "black", fill = NA, linewidth = 1),
      axis.title = element_text(size = 30),
      axis.text = element_text(size = 28, color = "black"),
      axis.title.x = element_text(margin = margin(t = 15)),
      legend.title = element_text(size = 30),
      legend.text = element_text(size = 28),
      legend.key.size = unit(1.5, "lines"),
      legend.key.spacing.y = unit(0.12, "cm"),
      legend.spacing.y = unit(0.6, "cm")
    )
}

p <- (pca_plot("PC1", "PC2") + theme(plot.margin = margin(r = 50)) | pca_plot("PC3", "PC4")) +
  plot_layout(guides = "collect")

options(repr.plot.width = 18, repr.plot.height = 6)
print(p)
save_fig(p, "Fig1D", 18, 6, 800)
write.csv(pc, file.path(SRC_DIR, "Fig1D_data.csv"), row.names = FALSE)
pct[1:4]

## Figure 3A

In [ ]:
plot_genes <- function(genes, file, ncol = 4, w = 5.8, h = 5.6) {
  d <- gene_counts(genes)
  d$gene <- factor(d$gene, levels = genes)
  d$x_time <- axis_time(d$time)
  nc <- min(ncol, length(genes))
  nr <- ceiling(length(genes) / ncol)
  pw <- w * nc + 3.6
  ph <- h * nr

  p <- ggplot(d, aes(x = x_time, y = count, color = infected, group = infected)) +
    geom_point(size = 3) +
    stat_summary(fun = "mean", geom = "line", linewidth = 1) +
    stat_summary(fun.data = mean_se, geom = "errorbar", linewidth = 1, width = 0.5) +
    frame_layer(genes, "gene", x_limits, 0.7) +
    scale_color_manual(values = c(no = palette[1], yes = palette[2]),
                       labels = c(no = "Control (LB)", yes = "MS2 (MOI = 10)")) +
    scale_x_continuous(limits = x_limits, breaks = x_breaks, labels = times, expand = expansion(0, 0)) +
    facet_wrap(~ gene, scales = "free_y", ncol = ncol) +
    labs(x = "Time (minutes post infection)", y = "Normalized count", color = "Infection") +
    theme_minimal() +
    theme(
      panel.border = element_blank(),
      panel.grid.minor.x = element_blank(),
      strip.background = element_blank(),
      panel.spacing.x = unit(1.0, "cm"),
      panel.spacing.y = unit(0.8, "cm"),
      axis.title.x = element_text(size = 30, margin = margin(t = 18)),
      axis.title.y = element_text(size = 30, margin = margin(r = 18)),
      axis.text = element_text(size = 26, color = "black"),
      axis.ticks = element_line(color = "black", linewidth = 0.7),
      axis.ticks.length = unit(0.3, "cm"),
      strip.text = element_text(face = "italic", size = 28),
      legend.title = element_text(size = 30),
      legend.key.size = unit(1.0, "cm"),
      legend.key.spacing.y = unit(0.2, "cm"),
      legend.text = element_text(size = 26, margin = margin(t = 2, b = 2, l = 6))
    ) +
    coord_cartesian(clip = "off")

  options(repr.plot.width = pw, repr.plot.height = ph)
  print(p)
  save_fig(p, file, pw, ph, 600)
  write.csv(d[, c("sample", "gene", "time", "infected", "count")],
            file.path(SRC_DIR, paste0(file, "_data.csv")), row.names = FALSE)
}

plot_genes(c("nhaR", "pgaA", "wcaE"), "Fig3A")

## Figure 4A

In [ ]:
tra_genes <- unique(c(grep("^(tra|trb)", rownames(dds), value = TRUE, ignore.case = TRUE), "ylpA", "finO"))
plasmid_genes <- c("tnpX", "tnpR", "tnpA", "pdeL_2", "yuaC", "yuaD", "yuaE", "yuaF", "LELOBEKK_04401", "insF7",
                   "yuaH", "insD8", "yuaK", "yuaM", "yuaN", "yuaO", "yuaP", "yuaQ", "ompP", "caeA", "yuaS",
                   "ccdA", "ccdB", "xerD_3", "repE", "sopB", "yubA", "yubB", "yhdJ_2", "yubE", "yubG", "klcA_3",
                   "ssb_2", "yubL", "noc", "psiB", "psiA", "flmC", "yubO", "yubP", "yubQ", "vapC", "vapB",
                   "srnB", "LELOBEKK_04479", "LELOBEKK_04429", "LELOBEKK_04430", "LELOBEKK_04427",
                   "LELOBEKK_04425", "LELOBEKK_04419", "LELOBEKK_04405", "LELOBEKK_04397", "LELOBEKK_04477")
plasmid_genes <- setdiff(unique(plasmid_genes), tra_genes)

modules <- list("Pilus-associated genes" = intersect(tra_genes, rownames(norm_counts)),
                "Non-pilus plasmid genes" = intersect(plasmid_genes, rownames(norm_counts)))
sapply(modules, length)
setdiff(plasmid_genes, rownames(norm_counts))

d <- do.call(rbind, lapply(names(modules), function(m) {
  data.frame(sample = colnames(norm_counts), module = m, count = colMeans(norm_counts[modules[[m]], , drop = FALSE]),
             time = meta$time, infected = meta$infected)
}))
d$module <- factor(d$module, levels = names(modules))
d$x_time <- axis_time(d$time)

p <- ggplot(d, aes(x = x_time, y = count, color = infected, group = infected)) +
  geom_hline(yintercept = 0, linetype = "dashed", color = "gray40", linewidth = 0.6) +
  geom_point(size = 2.5) +
  stat_summary(fun = "mean", geom = "line", linewidth = 0.8) +
  stat_summary(fun.data = mean_se, geom = "errorbar", linewidth = 0.8, width = 0.5) +
  frame_layer(names(modules), "module", x_limits, 0.6) +
  scale_color_manual(values = c(no = palette[1], yes = palette[2]),
                     labels = c(no = " Control (LB)", yes = " MS2 (MOI=10)")) +
  scale_x_continuous(limits = x_limits, breaks = x_breaks, labels = times, expand = expansion(0, 0)) +
  scale_y_continuous(limits = c(0, NA), expand = expansion(mult = c(0.08, 0.05)),
                     breaks = function(l) { b <- pretty(l); b[b >= 0] }) +
  facet_wrap(~ module, scales = "free_y", ncol = 4) +
  labs(x = "Time (minutes post infection)", y = "Module score", color = "Infection") +
  theme_minimal() +
  theme(
    panel.border = element_blank(),
    panel.grid.minor.x = element_blank(),
    strip.background = element_blank(),
    panel.spacing.x = unit(0.8, "cm"),
    axis.title.x = element_text(size = 26, margin = margin(t = 15)),
    axis.title.y = element_text(size = 26, margin = margin(r = 15)),
    axis.text = element_text(size = 22, color = "black"),
    axis.ticks = element_line(color = "black", linewidth = 0.8),
    axis.ticks.length = unit(0.35, "cm"),
    strip.text = element_text(size = 23),
    legend.title = element_text(size = 26),
    legend.key.spacing.y = unit(0.15, "cm"),
    legend.text = element_text(size = 23, margin = margin(t = 2, b = 2))
  ) +
  coord_cartesian(clip = "off")

pw <- 5.6 * 2 + 2.8
ph <- 5.4
options(repr.plot.width = pw, repr.plot.height = ph)
print(p)
save_fig(p, "Fig4A", pw, ph, 600)
write.csv(d[, c("sample", "module", "time", "infected", "count")], file.path(SRC_DIR, "Fig4A_data.csv"), row.names = FALSE)

### Module statistics reported in the text

In [ ]:
module_fold <- function(genes, g1, t1, g2, t2) {
  s <- colMeans(norm_counts[intersect(genes, rownames(norm_counts)), , drop = FALSE])
  a <- s[meta$infected == g1 & meta$time == t1]
  b <- s[meta$infected == g2 & meta$time == t2]
  data.frame(fold = mean(a) / mean(b), p = t.test(a, b)$p.value)
}

tra <- modules[[1]]
plasmid <- modules[[2]]
module_stats <- rbind(
  "tra inf vs ctrl 45"       = module_fold(tra,     "yes", 45,  "no",  45),
  "non-tra inf vs ctrl 45"   = module_fold(plasmid, "yes", 45,  "no",  45),
  "traA inf vs ctrl 45"      = module_fold("traA",  "yes", 45,  "no",  45),
  "tra inf 180 vs 45"        = module_fold(tra,     "yes", 180, "yes", 45),
  "non-tra inf 180 vs 45"    = module_fold(plasmid, "yes", 180, "yes", 45),
  "traA inf 180 vs 45"       = module_fold("traA",  "yes", 180, "yes", 45),
  "tra ctrl 180 vs 45"       = module_fold(tra,     "no",  180, "no",  45),
  "non-tra ctrl 180 vs 45"   = module_fold(plasmid, "no",  180, "no",  45),
  "tra inf 180 vs 0"         = module_fold(tra,     "yes", 180, "yes", 0)
)
module_stats$fold <- round(module_stats$fold, 2)
module_stats$p <- signif(module_stats$p, 2)
write.csv(module_stats, file.path(OUT_DIR, "Fig4A_module_stats.csv"))
module_stats

## Figure S11

In [ ]:
tra_order <- c("traM", "traJ", "traY", "traA", "traL", "traE", "traK", "traB", "traP",
               "trbD", "trbG", "traV", "traC", "trbI", "traW", "traU", "trbC", "traN", "trbE", "traF", "trbA",
               "traQ", "trbB", "trbJ", "trbF", "traH", "traG", "traS", "ylpA", "traD", "traI", "traX", "finO")
gene_labels <- setNames(tra_order, tra_order)
gene_labels["ylpA"] <- "traT"

d <- gene_counts(tra_order)
d$gene <- factor(gene_labels[d$gene], levels = gene_labels)
d$x_time <- axis_time(d$time)

ncol <- 6
nr <- ceiling(length(tra_order) / ncol)
n_empty <- nr * ncol - length(tra_order)
pw <- 3 * ncol + 1.0
ph <- 2.6 * nr + 1.4

p <- ggplot(d, aes(x = x_time, y = count, color = infected, group = infected)) +
  geom_point(size = 1.5) +
  stat_summary(fun = "mean", geom = "line", linewidth = 0.6) +
  stat_summary(fun.data = mean_se, geom = "errorbar", linewidth = 0.6, width = 0.4) +
  frame_layer(unname(gene_labels), "gene", x_limits, 0.5) +
  scale_color_manual(values = c(no = palette[1], yes = palette[2]),
                     labels = c(no = " control (LB)", yes = " MS2 (MOI=10)")) +
  guides(color = guide_legend(byrow = TRUE, override.aes = list(size = 5, linewidth = 1.6))) +
  scale_x_continuous(limits = x_limits, breaks = x_breaks, labels = times, expand = expansion(0, 0)) +
  facet_wrap(~ gene, scales = "free_y", ncol = ncol) +
  labs(x = "Time (minutes post infection)", y = "Normalized count", color = "Infection") +
  theme_minimal() +
  theme(
    panel.border = element_blank(),
    plot.margin = margin(t = 5.5, r = 25, b = 5.5, l = 5.5),
    panel.grid.minor.x = element_blank(),
    strip.background = element_blank(),
    panel.spacing.x = unit(0.4, "cm"),
    panel.spacing.y = unit(0.3, "cm"),
    axis.title.x = element_text(size = 28, margin = margin(t = 15)),
    axis.title.y = element_text(size = 28, margin = margin(r = 15)),
    axis.text.x = element_text(size = 12, color = "black"),
    axis.text.y = element_text(size = 16, color = "black"),
    axis.ticks = element_line(color = "black", linewidth = 0.6),
    axis.ticks.length = unit(0.18, "cm"),
    strip.text = element_text(size = 19),
    legend.title = element_text(size = 29, margin = margin(b = 12)),
    legend.title.position = "top",
    legend.position = if (n_empty > 0) "inside" else "right",
    legend.position.inside = c((ncol - n_empty / 2) / ncol, 0.5 / nr - 0.012),
    legend.justification = c(0.5, 0.5),
    legend.key.spacing.y = unit(0.7, "cm"),
    legend.key.height = unit(1.0, "cm"),
    legend.key.width = unit(1.7, "cm"),
    legend.text = element_text(size = 25, margin = margin(l = 8))
  ) +
  coord_cartesian(clip = "off")

options(repr.plot.width = pw, repr.plot.height = ph)
print(p)
save_fig(p, "FigS11", pw, ph, 300)
write.csv(d[, c("sample", "gene", "time", "infected", "count")], file.path(SRC_DIR, "FigS11_data.csv"), row.names = FALSE)

## DESeq2 comparisons (input for Table S1, built in `bulk_RNAseq_2_TableS1.ipynb`)

In [ ]:
de_infection <- function(t) {
  m <- meta[meta$time == t, ]
  x <- DESeqDataSetFromMatrix(countData = counts[, m$sample], colData = m, design = ~ line + infected)
  x <- DESeq(x, quiet = TRUE)
  results(x, contrast = c("infected", "yes", "no"))
}

de_time <- function(inf, t1, t2) {
  m <- meta[meta$infected == inf & meta$time %in% c(t1, t2), ]
  m$time <- factor(m$time, levels = c(t1, t2))
  x <- DESeqDataSetFromMatrix(countData = counts[, m$sample], colData = m, design = ~ line + time)
  x <- DESeq(x, quiet = TRUE)
  results(x, contrast = c("time", t2, t1))
}

de <- list()
for (t in times) {
  de[[paste0("infected_vs_control_", t, "min")]] <- de_infection(t)
}
for (inf in c("yes", "no")) {
  grp <- ifelse(inf == "yes", "infected", "control")
  for (i in seq_along(times)) for (j in seq_along(times)) if (i < j) {
    de[[paste0(grp, "_", times[j], "min_vs_", times[i], "min")]] <- de_time(inf, times[i], times[j])
  }
}

for (n in names(de)) {
  r <- as.data.frame(de[[n]])
  write.csv(r[order(r$padj), ], file.path(DE_DIR, paste0(n, ".csv")))
}